In [ ]:
# Sesión 5 del módulo 9, 08/09/2025
# 0) Instalación y arranque de Spark en Colab
# --- Colab: instalar PySpark (si no está) y crear una SparkSession ---
!pip -q install pyspark

from pyspark.sql import SparkSession

# Crea/obtiene la sesión de Spark
spark = SparkSession.builder.appName("Sesion5_MLlib").getOrCreate()

spark

In [ ]:
# 1) Regresión Logística con MLlib (Machine Learning Library): biblioteca de aprendizaje automático de Apache Spark.
 # (dataset Iris, flujo completo)
# Basado en los pasos de la guía: cargar, preprocesar (VectorAssembler + indexación), split, entrenar, predecir y evaluar
# ===========================
# REGRESIÓN LOGÍSTICA (IRIS)
# ===========================

# 1) Cargar/crear el dataset Iris
#    Si no tienes iris.csv, lo generamos desde scikit-learn y lo guardamos en /content/iris.csv
import os, pandas as pd
from sklearn import datasets

csv_path = "/content/iris.csv"
if not os.path.exists(csv_path):
    iris = datasets.load_iris()
    df_pd = pd.DataFrame(
        iris.data, columns=["sepal_length","sepal_width","petal_length","petal_width"]
    )
    # Agregamos la especie en texto para imitar el dataset clásico
    df_pd["species"] = pd.Series(iris.target).map({0:"setosa",1:"versicolor",2:"virginica"})
    df_pd.to_csv(csv_path, index=False)
# Condicional: si el archivo no existe, lo creamos.
# datasets.load_iris(): devuelve un “bunch” con data (matriz NumPy), target (0,1,2) y metadatos.
# pd.DataFrame(...): arma un DataFrame con 4 columnas numéricas. Les ponemos nombres claros: sepal_length, sepal_width, petal_length, petal_width.
# Agregar species (texto): el target de scikit-learn viene como enteros 0/1/2. Los mapeamos a nombres:
# 0 → setosa
# 1 → versicolor
# 2 → virginica
# Esto es útil porque en la guía luego convertimos este texto a etiqueta numérica con StringIndexer (parte del preprocesamiento sugerido)
# GUÍA ESTUDIO DE NUESTRA SESIÓN 5
# to_csv(..., index=False): guarda el archivo sin la columna de índice.
# Ventaja didáctica: el notebook queda autosuficiente. Si el CSV está, lo usa; si no, lo crea.

# Leemos el CSV con Spark (inferSchema=True detecta tipos numéricos)
data = spark.read.csv(csv_path, header=True, inferSchema=True)
# spark.read.csv crea un DataFrame de Spark desde el CSV.
# header=True: usa la primera fila como nombres de columna.
# inferSchema=True: Spark analiza valores y detecta tipos (por ejemplo, DoubleType para las medidas). Si lo quitaras, todo entraría como StringType,
# y tendrías que convertir tipos manualmente.

# Miramos 5 filas
data.show(5, truncate=False)
data.printSchema()
# show(5, truncate=False): imprime 5 filas sin cortar cadenas (útil para ver completo species).
# printSchema(): muestra el tipo de cada columna (deberías ver DoubleType para las 4 medidas y StringType para species).

# ¿Por qué guardar CSV si ya está en memoria con scikit-learn?
# Porque tu curso es de Big Data con Spark: queremos simular un fichero externo que Spark lee como lo haría en un entorno real (HDFS, S3, etc.).
# Así además practican spark.read.csv(...).
# Errores comunes
# Quitar inferSchema=True y luego fallar con VectorAssembler (“requires numeric types”).
# Encabezados mal puestos (header=False por accidente) → Spark interpreta la primera fila como datos.
# Rutas incorrectas (en Colab debe ser /content/...).

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|5.1         |3.5        |1.4         |0.2        |setosa |
|4.9         |3.0        |1.4         |0.2        |setosa |
|4.7         |3.2        |1.3         |0.2        |setosa |
|4.6         |3.1        |1.5         |0.2        |setosa |
|5.0         |3.6        |1.4         |0.2        |setosa |
+------------+-----------+------------+-----------+-------+
only showing top 5 rows

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)



Spark está mostrando un DataFrame estilo tabla.

Cada fila corresponde a una flor Iris.

Columnas:

sepal_length: longitud del sépalo (cm).

sepal_width: ancho del sépalo (cm).

petal_length: longitud del pétalo (cm).

petal_width: ancho del pétalo (cm).

species: la especie en texto (setosa en estos primeros ejemplos).

El mensaje only showing top 5 rows significa que solo imprime las 5 primeras filas (no todo el dataset completo, que tiene 150 registros).

Observa que todas estas filas corresponden a la especie Setosa. El dataset está ordenado por clases: primero Setosa, luego Versicolor, luego Virginica.

2. Resultado de data.printSchema()
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)


Aquí Spark te muestra el esquema del DataFrame.

Para cada columna indica:

Nombre (sepal_length, species, etc.).

Tipo de dato:

double: numérico decimal (los cuatro atributos).

string: texto (la especie).

nullable = true: significa que esa columna puede tener valores nulos (aunque en este dataset no los hay).

Interpretación completa:
Acabas de cargar correctamente el dataset Iris en un DataFrame de Spark. Tienes 4 columnas numéricas (double) y 1 categórica (species en texto). Este es exactamente el formato esperado para pasar al siguiente paso de la guía:

Convertir la columna species en un número (label).

Unir las 4 columnas numéricas en un solo vector (features).

In [ ]:
# 2) Preprocesamiento
#    a) Convertir "species" (string) a etiqueta numérica "label" (0,1,2)
from pyspark.ml.feature import StringIndexer
label_indexer = StringIndexer(inputCol="species", outputCol="label")
data_indexed = label_indexer.fit(data).transform(data)
# StringIndexer: es una clase de Spark MLlib que convierte una columna categórica de texto en números enteros.
# inputCol="species": columna de entrada (texto con valores "setosa", "versicolor", "virginica").
# outputCol="label": nombre de la columna de salida (la etiqueta numérica).
# fit(data): el indexador aprende el mapeo de texto → número según el orden de frecuencia (por defecto).
# Ejemplo: "setosa" → 0, "versicolor" → 1, "virginica" → 2.
# transform(data): aplica esa conversión al DataFrame, creando una nueva columna llamada label.
# Resultado intermedio (data_indexed):
# Ahora tu DataFrame tiene tanto la columna original species (texto) como la nueva columna label (numérica).

#    b) Unir las 4 características en un solo vector "features"
from pyspark.ml.feature import VectorAssembler
feature_cols = ["sepal_length","sepal_width","petal_length","petal_width"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data_feat = assembler.transform(data_indexed).select("features","label")
# VectorAssembler: transforma varias columnas numéricas en un único vector denso que Spark necesita para entrenar modelos.
# inputCols=feature_cols: las columnas que queremos combinar (las 4 medidas).
# outputCol="features": nombre de la nueva columna que contendrá el vector.
# transform(data_indexed): aplica la transformación y crea la columna features.
# .select("features","label"): nos quedamos solo con esas dos columnas (lo que Spark MLlib espera: features + label).
# Resultado final (data_feat):
# El DataFrame tiene 2 columnas:
# features: un vector con las 4 características [sepal_length, sepal_width, petal_length, petal_width].
# label: la especie ya codificada como número (0, 1, 2).

# Ejemplo de salida de data_feat.show(5, truncate=False):
# +-----------------+-----+
# |features         |label|
# +-----------------+-----+
# |[5.1,3.5,1.4,0.2]|0.0  |
# |[4.9,3.0,1.4,0.2]|0.0  |
# |[4.7,3.2,1.3,0.2]|0.0  |
# |[4.6,3.1,1.5,0.2]|0.0  |
# |[5.0,3.6,1.4,0.2]|0.0  |
# +-----------------+-----+
# only showing top 5 rows

data_feat.show(5, truncate=False)
# ¿Por qué es importante este paso?

# Los algoritmos de Spark MLlib NO trabajan con texto → por eso species debe transformarse a label.
# Los modelos esperan una sola columna features de tipo vector → por eso juntamos las 4 medidas.
# Ahora tenemos el formato estándar de Spark MLlib:
# features (vector de entrada)
# label (valor esperado)
# Y con esto ya podemos pasar al split train/test y luego a entrenar la Regresión Logística.

+-----------------+-----+
|features         |label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|0.0  |
|[4.9,3.0,1.4,0.2]|0.0  |
|[4.7,3.2,1.3,0.2]|0.0  |
|[4.6,3.1,1.5,0.2]|0.0  |
|[5.0,3.6,1.4,0.2]|0.0  |
+-----------------+-----+
only showing top 5 rows



Columna features

Es un vector denso (DenseVector) que contiene las 4 características de cada flor Iris:

sepal_length

sepal_width

petal_length

petal_width

Ejemplo:
[5.1, 3.5, 1.4, 0.2] → significa que esa flor tiene:

sépalo de 5.1 cm de largo,

sépalo de 3.5 cm de ancho,

pétalo de 1.4 cm de largo,

pétalo de 0.2 cm de ancho.

Columna label

Es la especie codificada en número, gracias a StringIndexer.

Aquí todas las filas tienen 0.0, lo que corresponde a la especie Setosa.

En el dataset completo, también habrá:

1.0 para Versicolor,

2.0 para Virginica.

Mensaje final

only showing top 5 rows → Spark solo muestra las 5 primeras filas, pero el DataFrame tiene 150 filas en total (todas las observaciones del dataset Iris).

Interpretación final:
Ya tienes el formato estándar que necesitan los algoritmos de MLlib:

features → vector con las variables independientes.

label → variable dependiente, en forma numérica.

Este dataset (data_feat) es el que usarás para:

Dividir en entrenamiento y prueba.

Entrenar la Regresión Logística.

Evaluar y hacer predicciones.

In [ ]:
# 3) División train/test
train, test = data_feat.randomSplit([0.8, 0.2], seed=42)
print("Tamaños -> train:", train.count(), " test:", test.count())

Tamaños -> train: 126  test: 24


In [ ]:
# 4) Modelo: Regresión Logística multiclase
from pyspark.ml.classification import LogisticRegression
# Estás trayendo la clase LogisticRegression de Spark MLlib (módulo de clasificación).
# Esta implementación soporta tanto clasificación binaria como multiclase (como el dataset Iris: 3 especies).
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
# featuresCol="features"
# Indicas que la columna con los vectores de entrada es "features" (creada antes con VectorAssembler).
# labelCol="label"
# Indicas que la columna con la etiqueta numérica es "label" (creada antes con StringIndexer).
# maxIter=50
# Número máximo de iteraciones que usará el algoritmo de optimización (método iterativo para ajustar los pesos del modelo).
# Si no lo pones, por defecto son 100, pero aquí lo limitamos a 50 para hacer el entrenamiento más rápido.
# En este punto, aún no has entrenado nada: solo configuraste el “molde” del modelo.

# Entrenar
lr_model = lr.fit(train)
# fit(train): entrena el modelo con el conjunto de datos de entrenamiento (train), que viene de la división train/test que hiciste antes.

# Spark aplica:
# Inicialización de los pesos (coeficientes de la regresión).
# Optimización iterativa (gradiente descendente u otro optimizador) para minimizar la función de pérdida (log loss).
# Ajuste de los coeficientes y bias que mejor predicen la label a partir de features.
# Resultado (lr_model): un objeto que contiene:
# Los parámetros aprendidos (pesos para cada variable).
# Funciones para hacer predicciones (transform()).
# Información sobre el entrenamiento (resumen, métricas, etc.).
#Antes teníamos datos organizados (features, label).
# Ahora le decimos a Spark:
# “Ajusta un modelo de regresión logística que pueda distinguir las especies de flores en función de sus medidas”.
# El algoritmo “aprende” los patrones matemáticos que mejor separan las clases.
# Con lr_model ya podemos pasar a:
# Hacer predicciones en el conjunto de prueba.
# Calcular métricas de evaluación (accuracy, F1, etc.).#

In [ ]:
# 5) Predicciones en test
pred = lr_model.transform(test)
# transform(test): aplica el modelo entrenado a los datos del conjunto de prueba (test).
# Spark añade nuevas columnas al DataFrame original:
# rawPrediction: valores en bruto antes de normalizar (log-odds).
# probability: vector con las probabilidades de cada clase.
# prediction: clase final predicha (la de mayor probabilidad).
# pred ahora es un DataFrame con columnas originales + resultados del modelo.
pred.select("features","label","prediction","probability").show(10, truncate=False)
# Aquí solo mostramos las columnas importantes para interpretar resultados:
# features → el vector de entrada (ej. [5.1,3.5,1.4,0.2]).
# label → la clase real (0.0 = Setosa, 1.0 = Versicolor, 2.0 = Virginica).
# prediction → la clase que el modelo predijo.
# probability → vector con la probabilidad estimada para cada clase.

+-----------------+-----+----------+----------------------------------------------------------------+
|features         |label|prediction|probability                                                     |
+-----------------+-----+----------+----------------------------------------------------------------+
|[4.4,3.0,1.3,0.2]|0.0  |0.0       |[1.0,5.804716249876623E-25,6.265523719986318E-48]               |
|[4.6,3.2,1.4,0.2]|0.0  |0.0       |[1.0,1.7180406186113684E-27,7.415660696413697E-51]              |
|[4.6,3.6,1.0,0.2]|0.0  |0.0       |[1.0,1.3750263699721883E-37,1.1836371120116384E-63]             |
|[4.8,3.1,1.6,0.2]|0.0  |0.0       |[1.0,1.6438553903861408E-23,5.216043961160273E-46]              |
|[4.9,3.1,1.5,0.1]|0.0  |0.0       |[1.0,8.603987252904992E-25,1.758079451753581E-48]               |
|[5.0,2.3,3.3,1.0]|1.0  |1.0       |[1.5277804846106156E-13,0.9999999926533467,7.346500420446885E-9]|
|[5.0,3.5,1.3,0.3]|0.0  |0.0       |[1.0,6.13841714409051E-31,3.1396115532173285E-

Columnas
features

Es el vector con las 4 medidas de cada flor (sepal_length, sepal_width, petal_length, petal_width).

label
Es la clase real de la flor (la especie, codificada numéricamente con StringIndexer):

0.0 → Setosa

1.0 → Versicolor

2.0 → Virginica

prediction

Es la clase que el modelo predijo.

probability

Es un vector con las probabilidades predichas para cada clase.

El orden corresponde a [probabilidad de clase 0, probabilidad de clase 1, probabilidad de clase 2].

Interpretación de ejemplos
Fila 1
features = [4.4,3.0,1.3,0.2]
label = 0.0
prediction = 0.0
probability = [1.0, 5.8E-25, 6.2E-48]


El modelo vio una flor muy típica de Setosa (label=0.0).

Predicción también fue 0.0 → ✔️ correcto.

Probabilidades: 100% a clase 0, prácticamente 0% a las otras.

Eso significa que el modelo está extremadamente seguro.

Fila 6
features = [5.0,2.3,3.3,1.0]
label = 1.0
prediction = 1.0
probability = [1.5E-13, 0.9999999926, 7.3E-9]


Clase real = 1.0 (Versicolor).

Predicción también = 1.0 → ✔️ correcto.

El modelo asignó ~99.999999% de probabilidad a la clase 1.

Es decir, está casi totalmente seguro de que es Versicolor.

Fila 10
features = [5.4,3.0,4.5,1.5]
label = 1.0
prediction = 1.0
probability = [6.9E-14, 0.996, 0.0037]


Clase real = 1.0 (Versicolor).

Predicción = 1.0 → ✔️ correcto.

El modelo asignó 99.6% de probabilidad a clase 1, y un 0.37% a Virginica.

Aquí hay un poco más de incertidumbre, pero igual predijo bien.

Conclusión

El modelo está funcionando correctamente:

label (verdadero) y prediction (estimado) coinciden en todos los casos que se ven.

Las probabilidades muestran el grado de confianza.

Valores como 1.0 o 0.9999 → predicción muy segura.

Valores con dos clases cercanas → predicción con más incertidumbre.

Esto es lo que permite evaluar si el modelo solo acierta de casualidad o si realmente está seguro de sus predicciones.

In [ ]:
# 6) Evaluación: accuracy, F1, matriz de confusión
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metric in ["accuracy","f1","weightedPrecision","weightedRecall"]:
    ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                           metricName=metric)
    print(f"{metric}: {ev.evaluate(pred):.4f}")

# (Opcional) Matriz de confusión simple
from pyspark.sql.functions import col
cm = (pred.groupBy("label","prediction").count()
           .orderBy(col("label"), col("prediction")))
cm.show()

accuracy: 1.0000
f1: 1.0000
weightedPrecision: 1.0000
weightedRecall: 1.0000
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   11|
|  1.0|       1.0|    6|
|  2.0|       2.0|    7|
+-----+----------+-----+



Estas métricas se calcularon con MulticlassClassificationEvaluator sobre el DataFrame pred.

Accuracy (exactitud):
Es el porcentaje de predicciones correctas.

1.0000 significa 100% de aciertos (todas las predicciones fueron correctas).

F1-score:
Media armónica entre precisión y recall, muy usada para clasificación.

1.0000 indica que el modelo no solo acertó, sino que lo hizo de forma perfecta en balance entre “no generar falsos positivos” y “no dejar pasar falsos negativos”.

Weighted Precision (precisión ponderada):
Medida de qué tan “limpias” son las predicciones por clase, ponderada por el número de instancias de cada clase.

1.0000 → todas las instancias clasificadas como una clase eran realmente de esa clase.

Weighted Recall (recobrado ponderado):
Qué proporción de ejemplos de cada clase se identificaron correctamente, ponderado por la frecuencia de esa clase.

1.0000 → el modelo detectó todas las instancias de cada clase.

En resumen: el modelo tuvo un desempeño perfecto en este conjunto de prueba.

Matriz de confusión
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   11|
|  1.0|       1.0|    6|
|  2.0|       2.0|    7|
+-----+----------+-----+


Esta tabla muestra, para cada combinación de etiqueta real (label) y predicción del modelo (prediction), cuántos casos se dieron.

Columnas:

label → clase real (0=Setosa, 1=Versicolor, 2=Virginica).

prediction → clase predicha por el modelo.

count → número de instancias en esa combinación.

Interpretación:

Fila 1: 11 flores eran realmente clase 0.0 (Setosa) y el modelo las predijo todas como 0.0.

Fila 2: 6 flores eran 1.0 (Versicolor) y fueron clasificadas como 1.0.

Fila 3: 7 flores eran 2.0 (Virginica) y todas fueron clasificadas como 2.0.

No aparecen otras combinaciones (como label=0.0 y prediction=1.0) porque no hubo errores.

Conclusión final

El modelo clasificó a la perfección todas las flores en el conjunto de prueba.

Esto se refleja en:

Métricas = 1.0 (perfecto).

Matriz de confusión diagonal (todos los aciertos están en la diagonal principal, y no hay errores fuera de ella).

In [ ]:
# 2) Clustering con K-Means (no supervisado, k=3)
# =====================
# K-MEANS (NO SUPERV.)
# Aquí ya no usamos etiquetas (label) porque es un algoritmo no supervisado.
# El objetivo es agrupar los datos automáticamente en clústeres (grupos), en este caso k=3 (porque sabemos que el dataset Iris tiene 3 especies).
# =====================

from pyspark.ml.clustering import KMeans
# Cargamos la clase KMeans desde la librería de MLlib en Spark.

# Usamos el mismo DataFrame "data_feat" con la columna "features"
kmeans = KMeans(featuresCol="features", k=3, seed=42)
# featuresCol="features": indicamos que usaremos la columna features (el vector de 4 medidas).
# k=3: número de clústeres (grupos) que queremos encontrar. Aquí 3 = Setosa, Versicolor, Virginica (aunque K-Means no sabe sus nombres).
# seed=42: semilla para que los resultados sean reproducibles (si no, cada ejecución puede dar centros distintos).
kmeans_model = kmeans.fit(data_feat)
# Entrenar el modelo, fit(data_feat): Spark aplica el algoritmo de K-Means para agrupar las flores en 3 clústeres.
# Calcula centros de clúster (centroids) que representan a cada grupo.

clusters = kmeans_model.transform(data_feat)
# Asignar cada dato a un clúster.
# transform(...): agrega al DataFrame una columna nueva llamada prediction, que indica el clúster al que pertenece cada flor (0, 1 o 2).

# Mostrar las primeras asignaciones de clúster
clusters.select("features", "prediction").show(10, truncate=False)
# Ver asignaciones de clúster
# Muestra las primeras 10 filas:
# features: vector con las medidas.
# prediction: número del clúster asignado (ej. 0, 1, o 2).
# Ejemplo típico:
# +-----------------+----------+
# |features         |prediction|
# +-----------------+----------+
# |[5.1,3.5,1.4,0.2]|0         |
# |[6.2,3.4,5.4,2.3]|2         |
# |[5.9,3.0,4.2,1.5]|1         |
# +-----------------+----------+


# Centros de los clústeres
centers = kmeans_model.clusterCenters()
print("\nCentros de los 3 clústeres:")
for i, c in enumerate(centers):
    print(f"  Centro {i}: {c}")
# clusterCenters() devuelve los vectores que representan el “promedio” de cada clúster.
# Ejemplo de salida:

# Centros de los 3 clústeres:
# Centro 0: [5.0,3.4,1.5,0.2]
# Centro 1: [6.0,2.8,4.5,1.3]
# Centro 2: [6.8,3.0,5.9,2.1]
# Cada centro es el perfil promedio de las flores en ese grupo.

# (Opcional) Calcular SSE (Within Set Sum of Squared Errors)
print("\nSSE (Within Set SSE):", kmeans_model.summary.trainingCost)
# SSE (Sum of Squared Errors): mide qué tan compactos son los clústeres.
# Mientras más bajo sea el valor, mejor los puntos están agrupados alrededor de sus centros.
# Se usa para comparar modelos con distinto k y decidir el número adecuado de clústeres (método del codo).

+-----------------+----------+
|features         |prediction|
+-----------------+----------+
|[5.1,3.5,1.4,0.2]|0         |
|[4.9,3.0,1.4,0.2]|0         |
|[4.7,3.2,1.3,0.2]|0         |
|[4.6,3.1,1.5,0.2]|0         |
|[5.0,3.6,1.4,0.2]|0         |
|[5.4,3.9,1.7,0.4]|0         |
|[4.6,3.4,1.4,0.3]|0         |
|[5.0,3.4,1.5,0.2]|0         |
|[4.4,2.9,1.4,0.2]|0         |
|[4.9,3.1,1.5,0.1]|0         |
+-----------------+----------+
only showing top 10 rows


Centros de los 3 clústeres:
  Centro 0: [5.006 3.428 1.462 0.246]
  Centro 1: [6.85384615 3.07692308 5.71538462 2.05384615]
  Centro 2: [5.88360656 2.74098361 4.38852459 1.43442623]

SSE (Within Set SSE): 78.85566582597728


1. Primer bloque: asignaciones de clúster
+-----------------+----------+
|features         |prediction|
+-----------------+----------+
|[5.1,3.5,1.4,0.2]|0         |
|[4.9,3.0,1.4,0.2]|0         |
|[4.7,3.2,1.3,0.2]|0         |
|[4.6,3.1,1.5,0.2]|0         |
|[5.0,3.6,1.4,0.2]|0         |
|[5.4,3.9,1.7,0.4]|0         |
|[4.6,3.4,1.4,0.3]|0         |
|[5.0,3.4,1.5,0.2]|0         |
|[4.4,2.9,1.4,0.2]|0         |
|[4.9,3.1,1.5,0.1]|0         |
+-----------------+----------+
only showing top 10 rows


features → vector de las 4 medidas (sepal_length, sepal_width, petal_length, petal_width).

prediction → el número de clúster asignado por K-Means (aquí todos son 0).

Todas estas primeras filas corresponden a Setosa, que el algoritmo agrupó en el clúster 0.
Más adelante en el dataset, las flores de Versicolor y Virginica aparecerán clasificadas en los otros clústeres (1 y 2).

2. Centros de los clústeres
Centros de los 3 clústeres:
  Centro 0: [5.006 3.428 1.462 0.246]
  Centro 1: [6.85384615 3.07692308 5.71538462 2.05384615]
  Centro 2: [5.88360656 2.74098361 4.38852459 1.43442623]


Cada centroide es el “promedio” de las características de las flores dentro de un clúster:

Centro 0: valores pequeños de pétalo y sépalo → corresponde a Setosa.

Centro 1: valores grandes (pétalo largo y ancho) → corresponde a Virginica.

Centro 2: valores intermedios → corresponde a Versicolor.

Esto refleja que K-Means logró separar las 3 especies aunque nunca vio la columna label.

3. Métrica de calidad (SSE)
SSE (Within Set SSE): 78.85566582597728


SSE (Sum of Squared Errors) mide la “compacidad” de los clústeres:

Calcula la distancia de cada punto a su centroide y las suma al cuadrado.

Un SSE más bajo significa clústeres más ajustados.

Aquí el valor ~78.86 es razonable, porque Iris tiene 3 grupos bien definidos.

Conclusión

Las primeras 10 flores fueron todas clasificadas en el clúster 0 (Setosa).

Los centros reflejan las medias de cada especie: Setosa (clúster 0), Virginica (clúster 1), Versicolor (clúster 2).

El SSE confirma que la agrupación es buena, ya que los clústeres son compactos y separados.

In [ ]:
# 3) Ejemplo genérico estilo lámina: datos.csv + LogisticRegression
# Replica el snippet minimalista de la presentación, pero creando datos.csv si no existe (dos features + label binaria).
# ======================================================
# EJEMPLO GENÉRICO: datos.csv con 2 features + label
# ======================================================

# 1) Generar un CSV sintético si no existe
import numpy as np, pandas as pd, os
rng = np.random.default_rng(7)
# numpy: para generar números aleatorios.
# pandas: para armar un DataFrame y exportarlo a CSV.
# os: para verificar si el archivo ya existe.

# rng = np.random.default_rng(7): inicializa un generador de números aleatorios con semilla 7 (así siempre se crean los mismos datos, lo que
# da reproducibilidad).

csv_path = "/content/datos.csv"
# Se guardará el dataset en /content/datos.csv (ubicación típica de Google Colab).
if not os.path.exists(csv_path):
    n = 200
    f1 = rng.normal(0, 1, n)
    f2 = rng.normal(0, 1, n)
# if not os.path.exists(csv_path) → si el archivo no existe, lo generamos desde cero.
# n = 200: vamos a crear 200 observaciones.
# f1 y f2: dos variables simuladas, cada una con distribución normal (media 0, desviación estándar 1).
# Estas son nuestras dos “features”.

    # Regla simple para la etiqueta binaria (lineal + ruido)
    y = (0.8*f1 + 1.2*f2 + rng.normal(0, 0.6, n) > 0).astype(int)
# Construimos una regla lineal con algo de ruido:
# 0.8*f1 + 1.2*f2: combinación lineal de las dos variables.
# + rng.normal(0, 0.6, n): agregamos ruido gaussiano para que no sea 100% determinístico.
# > 0: si la suma es positiva → clase 1, si no → clase 0.
# astype(int): convierte el resultado booleano (True/False) en 1 o 0.
# Así generamos una variable binaria (label), con valores 0 y 1.
    pd.DataFrame({"feature1":f1, "feature2":f2, "label":y}).to_csv(csv_path, index=False)
# Creamos un DataFrame con 3 columnas:
# feature1
# feature2
# label
# Lo guardamos como datos.csv sin columna de índice (index=False).
# Ejemplo de cómo se vería el archivo:

# feature1,feature2,label
# -0.12,  0.85, 1
#  1.34, -0.45, 0
# -0.88,  1.20, 1

# En resumen
# Este bloque genera un dataset sintético simple de 200 observaciones con:
# Dos características continuas (feature1, feature2).
# Una etiqueta binaria (label = 0 o 1).
# Luego lo guarda en un archivo CSV (datos.csv) que Spark puede leer y usar para entrenar un modelo de regresión logística binaria.

In [ ]:
# 2) Cargar con Spark
data = spark.read.csv(csv_path, header=True, inferSchema=True)
# Le estás diciendo a Spark: “lee un archivo CSV y conviértelo en un DataFrame distribuido”.
# Este es el equivalente a pd.read_csv(...) en pandas, pero en Spark los datos quedan listos para procesarse en paralelo en clústeres
# grandes (aunque aquí estamos en Colab/local).

# Parámetro csv_path
# Es la ruta del archivo que creaste en el paso anterior (/content/datos.csv).
# Si lo abrieras en Colab verías algo como:
# feature1,feature2,label
# -0.12,0.85,1
# 1.34,-0.45,0
# -0.88,1.20,1

# header=True
# Le dice a Spark que la primera fila del CSV contiene los nombres de las columnas (feature1, feature2, label).
# Si lo pones en False, Spark pondría nombres genéricos como _c0, _c1, _c2.

# inferSchema=True
# Spark intenta adivinar automáticamente el tipo de dato de cada columna:
# feature1, feature2 → DoubleType (números decimales).
# label → IntegerType (0 o 1).
# Si no lo pones, todo se leería como texto (StringType) y luego habría que convertirlo manualmente.
# Con inferSchema=True puedes usar directamente estas columnas en un modelo de ML sin transformaciones extra.

# Asignación a data
# El resultado se guarda en un DataFrame de Spark llamado data.
# Puedes inspeccionarlo con:
# data.show(5)
# data.printSchema()
# Ejemplo de salida:
# +---------+---------+-----+
# | feature1| feature2|label|
# +---------+---------+-----+
# |   -0.120|    0.850|    1|
# |    1.340|   -0.450|    0|
# |   -0.880|    1.200|    1|
# +---------+---------+-----+
# only showing top 3 rows

# root
 # |-- feature1: double (nullable = true)
 # |-- feature2: double (nullable = true)
 # |-- label: integer (nullable = true)

# En resumen
# Esa línea:
# Lee el archivo datos.csv.
# Reconoce las columnas (feature1, feature2, label).
# Detecta que son numéricas.
# Devuelve un DataFrame de Spark (data) listo para usarse en VectorAssembler y LogisticRegression.


In [ ]:
# 3) VectorAssembler -> "features"
# Esa parte del código es la que prepara los datos para que Spark MLlib pueda usarlos en un modelo de Machine Learning.
from pyspark.ml.feature import VectorAssembler
# VectorAssembler es una utilidad de Spark MLlib que combina varias columnas numéricas en un solo vector.
# Los modelos de MLlib (como LogisticRegression) esperan que todas las variables de entrada estén en una única columna tipo vector, llamada
# normalmente "features".
assembler = VectorAssembler(inputCols=["feature1","feature2"], outputCol="features")
# inputCols=["feature1","feature2"]
# Indica qué columnas queremos unir en un vector.
# En este caso, las dos variables independientes que creamos: feature1 y feature2.
# outputCol="features"
# Nombre de la nueva columna donde se guardará el vector combinado.
# Cada fila tendrá un vector como [f1, f2].
data_feat = assembler.transform(data)
# transform(data) aplica la transformación al DataFrame data.
# Resultado: se añade una nueva columna "features".
# Ahora el DataFrame tiene: feature1, feature2, label y features.
# Ejemplo de salida con data_feat.show(5, truncate=False)
# +---------+---------+-----+---------------+
# |feature1 |feature2 |label|features       |
# +---------+---------+-----+---------------+
# | -0.120  |  0.850  |  1  |[-0.120,0.850] |
# |  1.340  | -0.450  |  0  |[1.340,-0.450] |
# | -0.880  |  1.200  |  1  |[-0.880,1.200] |
# |  0.250  |  0.100  |  0  |[0.250,0.100]  |
# | -1.100  | -0.750  |  0  |[-1.100,-0.750]|
# +---------+---------+-----+---------------+
# La nueva columna features contiene un vector [feature1, feature2].
# Esta es la columna que el modelo de regresión logística usará como entrada.

# En resumen
# Antes: teniamos dos columnas separadas (feature1, feature2).
# Después: ahora tenemos una sola columna features que contiene [feature1, feature2].
# Motivo: Spark MLlib requiere esta columna vectorizada para entrenar los modelos.

In [ ]:
# 4) Split
train, test = data_feat.randomSplit([0.8, 0.2], seed=123)
# data_feat
# Es tu DataFrame con:
# label → variable dependiente (0 o 1).
# features → vector con las variables independientes [feature1, feature2].
# Ya está listo para usarse en un modelo de MLlib.
# randomSplit([0.8, 0.2], seed=123)
# Divide aleatoriamente el DataFrame en dos subconjuntos:
# 80% de los datos → train (conjunto de entrenamiento).
# 0% de los datos → test (conjunto de prueba).

In [ ]:
# 5) Logistic Regression binaria
# Llegamos al paso en el que realmente construimos y entrenamos un modelo de clasificación con los datos.
from pyspark.ml.classification import LogisticRegression
# Cargas el algoritmo Regresión Logística desde el módulo de clasificación de Spark MLlib.
# Este algoritmo es muy usado para clasificación binaria (dos clases, aquí 0 y 1).
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
# featuresCol="features"
# Indicas que la columna de entrada es features (la que creaste con VectorAssembler que contiene [feature1, feature2]).
# labelCol="label"
# Indicas que la columna que contiene la clase real es label (0 o 1, generada en tu dataset).
# maxIter=20
# Es el número máximo de iteraciones del optimizador (el algoritmo que ajusta los coeficientes del modelo).
# En cada iteración, el modelo ajusta los pesos para minimizar el error.
# Si necesitas más precisión, puedes aumentar este número (ej. 100).
# En este punto, lr es solo la configuración del modelo, aún no ha aprendido nada.

# Entrenar el modelo
model = lr.fit(train)
# fit(train) → entrena el modelo usando los datos de entrenamiento (train).
# Spark ajusta los parámetros del modelo (coeficientes y bias) para encontrar la mejor separación entre las clases 0 y 1.
# El resultado (model) es un objeto que contiene:
# Los coeficientes aprendidos (pesos para cada feature).
# El intercepto (bias).
# Métodos para hacer predicciones (transform()).
# Información sobre métricas de entrenamiento.

In [ ]:
# 6) Predicción y vista rápida
pred = model.transform(test)
pred.select("label","prediction","probability").show(10, truncate=False)

+-----+----------+------------------------------------------+
|label|prediction|probability                               |
+-----+----------+------------------------------------------+
|0    |0.0       |[0.9966145481899489,0.003385451810051121] |
|0    |0.0       |[0.9978570918618561,0.002142908138143884] |
|0    |0.0       |[0.9795974757446678,0.020402524255332177] |
|0    |0.0       |[0.9862505073327709,0.013749492667229135] |
|0    |0.0       |[0.961162345389691,0.03883765461030897]   |
|1    |1.0       |[0.0039219506377551824,0.9960780493622449]|
|1    |0.0       |[0.9561112365154126,0.04388876348458737]  |
|0    |0.0       |[0.8066124903225133,0.19338750967748675]  |
|0    |0.0       |[0.8657420050510689,0.1342579949489311]   |
|0    |1.0       |[0.04547199501939668,0.9545280049806033]  |
+-----+----------+------------------------------------------+
only showing top 10 rows



Esa salida es el resultado de aplicar el modelo de regresión logística binaria a tu conjunto de prueba y mostrar tres columnas clave: label, prediction y probability.

La tabla
+-----+----------+------------------------------------------+
|label|prediction|probability                               |
+-----+----------+------------------------------------------+
|0    |0.0       |[0.9966,0.0033]                           |
|0    |0.0       |[0.9978,0.0021]                           |
|0    |0.0       |[0.9796,0.0204]                           |
|0    |0.0       |[0.9862,0.0137]                           |
|0    |0.0       |[0.9611,0.0388]                           |
|1    |1.0       |[0.0039,0.9960]                           |
|1    |0.0       |[0.9561,0.0438]                           |
|0    |0.0       |[0.8066,0.1933]                           |
|0    |0.0       |[0.8657,0.1342]                           |
|0    |1.0       |[0.0454,0.9545]                           |
+-----+----------+------------------------------------------+
only showing top 10 rows

Explicación de cada columna

label

La clase real del dato.

0 = clase negativa, 1 = clase positiva.

prediction

La clase predicha por el modelo.

Spark elige la clase con mayor probabilidad.

probability

Vector con las probabilidades estimadas por el modelo para cada clase.

Formato: [P(clase=0), P(clase=1)].

Ejemplo: [0.9966, 0.0033] significa que el modelo cree con 99.6% de seguridad que es clase 0.

Interpretación fila a fila

Fila 1:

label=0, prediction=0.0, probability=[0.9966,0.0033]

El modelo acertó → estaba 99.6% seguro de que era clase 0.

Fila 6:

label=1, prediction=1.0, probability=[0.0039,0.9960]

Acertó → 99.6% seguro de que era clase 1.

Fila 7 (error):

label=1, prediction=0.0, probability=[0.9561,0.0438]

El modelo falló → la flor era clase 1, pero el modelo dio 95.6% de probabilidad a clase 0.

Aquí se ve que ningún modelo es perfecto: cometió una clasificación errónea.

Fila 10 (otro error):

label=0, prediction=1.0, probability=[0.0454,0.9545]

El modelo pensó que era clase 1 con un 95% de confianza, pero en realidad era clase 0.

Conclusión

La mayoría de las predicciones coinciden con las etiquetas reales (label = prediction).

El vector de probability muestra cuán seguro estaba el modelo en cada caso.

Hay algunas clasificaciones erróneas (fila 7 y fila 10), lo que es normal en Machine Learning.

Justo por eso, en el siguiente paso se calculan métricas como accuracy, precision, recall o F1, para cuantificar el rendimiento global.

In [ ]:
# 7) Accuracy
# Esa parte del código es donde evaluamos qué tan bien funcionó la regresión logística binaria.
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
ev_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
print("Accuracy:", round(ev_acc.evaluate(pred), 4))

# (Opcional) AUC (solo binaria)
ev_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
print("AUC:", round(ev_auc.evaluate(pred), 4))

Accuracy: 0.9268
AUC: 0.9476


Accuracy = 0.9268

Significa que el modelo acertó en el 92.68% de las predicciones.

Es decir, de cada 100 ejemplos en el conjunto de prueba, aproximadamente 93 fueron clasificados correctamente y 7 fueron errores.

En clasificación binaria, una accuracy > 0.90 se considera muy buena, aunque depende del problema (si las clases estuvieran muy desbalanceadas, esta métrica sola no bastaría).

2. AUC = 0.9476

AUC = Area Under the ROC Curve (Área bajo la curva ROC).

Mide la capacidad del modelo de separar las dos clases en términos de sus probabilidades, no solo de aciertos directos.

El rango va de:

1.0 → separación perfecta.

0.5 → el modelo no distingue nada (equivalente a adivinar al azar).

Un AUC de 0.9476 (~95%) significa que el modelo es excelente para discriminar entre clase 0 y clase 1:

Si tomas un ejemplo positivo y uno negativo al azar, el 95% de las veces el modelo le asignará mayor probabilidad al positivo.

Conclusión

El modelo tiene un alto desempeño:

Muy buena exactitud (92.7%).

Muy buena capacidad de discriminación entre clases (AUC ≈ 95%).

Esto demuestra que la frontera de decisión que encontró la regresión logística se ajusta bien a los datos generados, aunque no es perfecta (hubo algunos errores de clasificación, como vimos en tu tabla de predicciones).

**Actividad**
ACTIVIDAD — SESIÓN: INTRODUCCIÓN A MACHINE LEARNING ESCALABLE (II)

Clasificación de productos capilares según tipo de cabello

Una tienda de cuidado capilar desea un sistema que clasifique shampoos y acondicionadores en categorías de tipo de cabello recomendado para personalizar recomendaciones.

Dataset (archivo: haircare_products.csv) con columnas:

Ingrediente principal (p. ej., keratina, argán, biotina, romero, coco, cafeína…)

Limpieza: Bajo, Medio, Alto

Nutrición: Bajo, Medio, Alto

pH (numérico)

Sulfatos: Sí/No

Silicona: Sí/No

Tipo de Cabello (target): Seco, Graso, Mixto, Dañado/Teñido

Objetivo: Entrenar un modelo con MLlib que prediga el tipo de cabello recomendado.

Importante (codificación de etiqueta):

Seco → 0

Graso → 1

Mixto → 2

Dañado/Teñido → 3

INSTRUCCIONES Y PUNTAJE
1) Carga y exploración de datos (2 pts)

Cargar haircare_products.csv en un DataFrame de PySpark.

Mostrar las primeras filas del dataset.

Resumen estadístico de variables numéricas (pH; y cualquier otra que sea numérica).

2) Preprocesamiento (2 pts)

Convertir Tipo de Cabello a valores numéricos (0,1,2,3) según el mapeo dado (no usar el orden de StringIndexer).

Transformar Limpieza y Nutrición:

Bajo → 0, Medio → 1, Alto → 2

Convertir Sulfatos y Silicona a binario:

Sí → 1, No → 0

Indexar Ingrediente con StringIndexer.

Unir todo en un vector features con VectorAssembler.

3) División y entrenamiento (3 pts)

Split 80% entrenamiento / 20% prueba.

Entrenar un Árbol de Decisión (sug.: maxDepth 5–7, maxBins ≥ nº de categorías de Ingrediente, p. ej. 256).

4) Predicción y evaluación (2 pts)

Mostrar predicciones sobre el conjunto de prueba.

Calcular accuracy con MulticlassClassificationEvaluator.

(Opcional recomendado) Matriz de confusión.

5) Análisis y mejoras (1 pt)

En 3–5 líneas: precisión obtenida y cómo mejorar (p. ej., RandomForest o GBT, ajuste de hiperparámetros, mejor codificación de Ingrediente, balanceo de clases).

In [ ]:
# Starter kit (PySpark, listo para Colab)
# A) (Opcional) Generar un CSV sintético
# --- Generador de dataset sintético: haircare_products.csv ---
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 300
# numpy y pandas para generar y tabular datos.
# rng = np.random.default_rng(42): fija semilla para que cada ejecución produzca el mismo dataset (reproducible).
# n = 300: número de filas (productos) a generar

ingreds = ["keratina","argan","biotina","romero","coco","cafeina","colageno","aguacate"]
limp = ["Bajo","Medio","Alto"]
nutr = ["Bajo","Medio","Alto"]
yn = ["Sí","No"]
cabello = ["Seco","Graso","Mixto","Dañado/Teñido"]
# Listas de posibles valores para:
# Ingrediente principal,
# niveles de Limpieza y Nutrición,
# flags Sí/No (Sulfatos, Silicona),
# y las clases de Tipo de Cabello (la etiqueta que luego queremos predecir).
# Nota: cabello no se usa directamente; la etiqueta se calcula con una regla (abajo).

def sample_row():
    ing = rng.choice(ingreds)
    li  = rng.choice(limp, p=[0.3,0.5,0.2])
    nu  = rng.choice(nutr, p=[0.2,0.5,0.3])
    ph  = np.round(rng.normal(5.5, 0.6), 2)  # pH típico 4.5-6.5
    sul = rng.choice(yn, p=[0.4,0.6])        # Sí/No
    sil = rng.choice(yn, p=[0.5,0.5])
# rng.choice(lista, p=...) elige un valor con probabilidades dadas:
# Limpieza: Bajo 30%, Medio 50%, Alto 20%.
# Nutrición: Bajo 20%, Medio 50%, Alto 30%.
# Sulfatos: Sí 40%, No 60%.
# Silicona: 50/50.
# pH: se toma de una normal con media 5.5 y desvío 0.6, redondeado a 2 decimales (rango típico ~4.5–6.5).

    # regla suave para tipo de cabello
    if (nu=="Alto" and sil=="Sí") or ing in ["argan","coco","aguacate"]:
        tc = "Seco"
    elif (li=="Alto" and sul=="Sí") or ing in ["romero","cafeina"]:
        tc = "Graso"
    elif ing in ["keratina","colageno"] or (nu=="Medio" and li=="Medio"):
        tc = "Dañado/Teñido"
    else:
        tc = "Mixto"
    return [ing, li, nu, ph, sul, sil, tc]
# Construye una relación realista pero sintética entre features y clase:
# Seco: nutrición Alto + silicona Sí, o ingredientes argan/coco/aguacate (más nutritivos).
# Graso: limpieza Alto + sulfatos Sí, o ingredientes romero/cafeina (estimulantes/clarificantes).
# Dañado/Teñido: ingredientes keratina/colageno o combinación Medio/Medio.
# Mixto: caso “resto”.
# Esto crea patrones para que el modelo pueda aprender, pero con algo de ruido por las distribuciones aleatorias.

# Generar todas las filas, tabular y guardar
rows = [sample_row() for _ in range(n)]
df = pd.DataFrame(rows, columns=["Ingrediente","Limpieza","Nutrición","pH","Sulfatos","Silicona","Tipo de Cabello"])
df.to_csv("/content/haircare_products.csv", index=False)
print("OK -> /content/haircare_products.csv creado con", len(df), "filas")
df.head()
# Crea n filas llamando a sample_row.
# Construye un DataFrame con columnas ya nombradas.
# Guarda en Colab (ruta /content/...), sin índice.
# Imprime confirmación y muestra las primeras filas.

OK -> /content/haircare_products.csv creado con 300 filas


,Ingrediente,Limpieza,Nutrición,pH,Sulfatos,Silicona,Tipo de Cabello
0,keratina,Medio,Alto,6.06,Sí,No,Dañado/Teñido
1,colageno,Medio,Alto,5.49,No,Sí,Seco
2,argan,Medio,Alto,5.78,Sí,No,Seco
3,aguacate,Bajo,Alto,5.47,No,Sí,Seco
4,keratina,Alto,Alto,5.29,No,Sí,Seco


OK -> /content/haircare_products.csv creado con 300 filas
Significa que el generador creó y guardó el CSV sintético en Colab con 300 registros.

Vista de las primeras filas (df.head())

yaml
Copiar código
Ingrediente  Limpieza  Nutrición   pH  Sulfatos  Silicona  Tipo de Cabello
0  keratina     Medio       Alto  6.06       Sí        No   Dañado/Teñido
1  colageno     Medio       Alto  5.49        No        Sí  Seco
2  argan        Medio       Alto  5.78       Sí        No   Seco
3  aguacate     Bajo        Alto  5.47        No        Sí  Seco
4  keratina     Alto        Alto  5.29        No        Sí  Seco
Qué significa cada columna
Ingrediente: ingrediente principal del producto (elegido aleatoriamente de una lista).

Limpieza / Nutrición: niveles categóricos (Bajo, Medio, Alto) muestreados con ciertas probabilidades.

pH: valor numérico simulado ~N(5.5, 0.6).

Sulfatos / Silicona: Sí/No.

Tipo de Cabello: etiqueta generada por una regla en el código (no es aleatoria pura). La regla (en orden) dice:

Si Nutrición = Alto y Silicona = Sí, o el ingrediente es argan/coco/aguacate → Seco

Si Limpieza = Alto y Sulfatos = Sí, o el ingrediente es romero/cafeina → Graso

Si el ingrediente es keratina/colageno, o (Nutrición=Medio y Limpieza=Medio) → Dañado/Teñido

Si nada de lo anterior → Mixto

Importante: la prioridad es por el orden. Si una fila cumple la condición 1, ya no evalúa las siguientes.

Por qué cada fila quedó así
Fila 0: keratina, Medio, Alto, …, Silicona=No

(1) no se cumple (Silicona≠Sí y el ingrediente no es argan/coco/aguacate)

(2) no se cumple

(3) sí (ingrediente = keratina) → Dañado/Teñido

Fila 1: colageno, Medio, Alto, …, Silicona=Sí

(1) sí (Nutrición=Alto y Silicona=Sí) → Seco

Aunque colágeno activaría (3), la condición (1) tiene prioridad.

Fila 2: argan, …

(1) sí (ingrediente ∈ {argan, coco, aguacate}) → Seco.

Fila 3: aguacate, …

(1) sí (ingrediente = aguacate) → Seco (también Nutrición=Alto y Silicona=Sí).

Fila 4: keratina, Alto, Alto, …, Silicona=Sí

(1) sí (Nutrición=Alto y Silicona=Sí) → Seco

De nuevo, aunque keratina coincide con (3), gana la condición (1).

Con esto ves que el dataset no es “ruido puro”: tiene patrones razonables para que el modelo pueda aprender, pero las probabilidades y combinaciones introducen variabilidad realista.

In [ ]:
# B) Cargar, explorar, preprocesar, entrenar y evaluar
# --- Spark session (si no la tienes) ---
!pip -q install pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, IntegerType
spark = SparkSession.builder.appName("HairCareML").getOrCreate()

# 1) CARGA Y EXPLORACIÓN
data = spark.read.csv("/content/haircare_products.csv", header=True, inferSchema=True)
data.show(5, truncate=False)
data.select("pH").describe().show()
# read.csv con header=True usa la primera fila como nombres; inferSchema=True intenta detectar tipos.
# show(5) enseña ejemplos.
# describe() da estadísticas de pH (count, mean, stddev, min, max) para revisar rangos/valores raros.

# 2) PREPROCESAMIENTO
# 2.1) label (Tipo de Cabello) -> mapeo EXACTO del enunciado
norm_tc = F.upper(F.trim(F.col("Tipo de Cabello")))
data1 = (data
    .withColumn("label",
        F.when(norm_tc=="SECO", 0)
         .when(norm_tc=="GRASO", 1)
         .when(norm_tc=="MIXTO", 2)
         .when((norm_tc=="DAÑADO/TEÑIDO") | (norm_tc=="DANADO/TEÑIDO"), 3)
         .otherwise(F.lit(None))
    )
    .withColumn("label", F.col("label").cast(DoubleType()))
)
# Normaliza a mayúsculas y trim para que “ seco ”, “Seco”, etc., se traten igual.
# Mapea exactamente como pide el enunciado: Seco→0, Graso→1, Mixto→2, Dañado/Teñido→3 (incluye variante sin “Ñ” por si hay errores de carga).
# label se castea a Double porque MLlib lo espera así.

# 2.2) Limpieza y Nutrición -> (Bajo=0, Medio=1, Alto=2)
# Categóricas ordinales → números
def map_bma(colname):
    c = F.upper(F.trim(F.col(colname)))
    return (F.when(c=="BAJO", 0.0)
             .when(c=="MEDIO", 1.0)
             .when(c=="ALTO", 2.0)
             .otherwise(F.lit(None).cast(DoubleType())))

data2 = (data1
    .withColumn("LimpiezaIndex",  map_bma("Limpieza"))
    .withColumn("NutricionIndex", map_bma("Nutrición"))
)
# Convierte Limpieza y Nutrición a 0/1/2 (Bajo/Medio/Alto).
# Queda claro el orden (ordinal), cosa que un indexador “ciego” no sabría.


# 2.3) Sulfatos / Silicona -> binario (Sí=1, No=0)
def map_yesno(colname):
    c = F.upper(F.trim(F.col(colname)))
    return (F.when((c=="SI") | (c=="SÍ"), 1.0)
             .when(c=="NO", 0.0)
             .otherwise(F.lit(None).cast(DoubleType())))

data3 = (data2
    .withColumn("SulfatosBin", map_yesno("Sulfatos"))
    .withColumn("SiliconaBin", map_yesno("Silicona"))
    .withColumn("pH", F.col("pH").cast(DoubleType()))
)
# Maneja “SI” y “SÍ” (con/sin tilde).
# Castea pH a Double (por si se leyó como string).

# 2.4) Ingrediente -> índice numérico (categórica)
from pyspark.ml.feature import StringIndexer, VectorAssembler
ingred_indexer = StringIndexer(inputCol="Ingrediente", outputCol="IngredienteIndex", handleInvalid="keep")
data4 = ingred_indexer.fit(data3).transform(data3)
# StringIndexer da un número a cada ingrediente distinto (categórica nominal).
# handleInvalid="keep" evita errores si aparece un valor nuevo en test (lo manda a una categoría “desconocida”).
# Nota: si tuvieras un texto libre con varios ingredientes, convendría RegexTokenizer + CountVectorizer en lugar de StringIndexer.

# 2.5) Ensamblar features
feature_cols = ["IngredienteIndex","LimpiezaIndex","NutricionIndex","pH","SulfatosBin","SiliconaBin"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data_feat = assembler.transform(data4).select("features","label")
data_feat.show(5, truncate=False)
# VectorAssembler junta todas las columnas numéricas en un vector features, que es el formato estándar para entrenar con MLlib.
# data_feat ya solo tiene features y label → ¡listo para split, entrenar y evaluar!

+-----------+--------+---------+----+--------+--------+---------------+
|Ingrediente|Limpieza|Nutrición|pH  |Sulfatos|Silicona|Tipo de Cabello|
+-----------+--------+---------+----+--------+--------+---------------+
|keratina   |Medio   |Alto     |6.06|Sí      |No      |Dañado/Teñido  |
|colageno   |Medio   |Alto     |5.49|No      |Sí      |Seco           |
|argan      |Medio   |Alto     |5.78|Sí      |No      |Seco           |
|aguacate   |Bajo    |Alto     |5.47|No      |Sí      |Seco           |
|keratina   |Alto    |Alto     |5.29|No      |Sí      |Seco           |
+-----------+--------+---------+----+--------+--------+---------------+
only showing top 5 rows

+-------+------------------+
|summary|                pH|
+-------+------------------+
|  count|               300|
|   mean| 5.478466666666666|
| stddev|0.5838165706648211|
|    min|              3.31|
|    max|              7.19|
+-------+------------------+

+--------------------------+-----+
|features                  |la

1) Carga y vista rápida (data.show(5))
Ingrediente | Limpieza | Nutrición | pH   | Sulfatos | Silicona | Tipo de Cabello
keratina      Medio       Alto       6.06    Sí         No         Dañado/Teñido
...


Son las primeras 5 filas del CSV tal cual entró.

Columnas:

Ingrediente (categórica nominal),

Limpieza y Nutrición (categóricas ordinales: Bajo/Medio/Alto),

pH (numérica),

Sulfatos / Silicona (Sí/No),

Tipo de Cabello (la etiqueta en texto).

2) Resumen estadístico de pH
count  300
mean   5.4785
stddev 0.5838
min    3.31
max    7.19


Confirma que tienes 300 registros y que pH es numérico.

Rango razonable para productos capilares (~4.5–6.5), con algunos extremos (3.31, 7.19) que puedes revisar si quieres limpiar outliers.

3) Dataset para ML (data_feat.show(5))
features                          | label
[4.0,1.0,2.0,6.06,1.0,0.0]          3.0
[2.0,1.0,2.0,5.49,0.0,1.0]          0.0
...


features es un vector en este orden exacto:
["IngredienteIndex","LimpiezaIndex","NutricionIndex","pH","SulfatosBin","SiliconaBin"]

Fila 1: [4.0, 1.0, 2.0, 6.06, 1.0, 0.0]

IngredienteIndex = 4.0 → índice numérico que aprende StringIndexer para “keratina” (el número depende de las frecuencias del dataset).

LimpiezaIndex = 1.0 → Medio

NutricionIndex = 2.0 → Alto

pH = 6.06

SulfatosBin = 1.0 → Sí

SiliconaBin = 0.0 → No

label = 3.0 → Dañado/Teñido (mapeo fijo que definimos: Seco=0, Graso=1, Mixto=2, Dañado/Teñido=3)

Fila 2: [2.0,1.0,2.0,5.49,0.0,1.0]

IngredienteIndex = 2.0 (para “colageno” en este fit),

Limpieza=Medio (1.0), Nutrición=Alto (2.0),

Sulfatos=No (0.0), Silicona=Sí (1.0),

label = 0.0 → Seco.

Nota: El índice de Ingrediente depende de la frecuencia en el dataset (propio de StringIndexer). Si quieres ver el mapeo índice→ingrediente que se usó:

# Mostrar el mapeo aprendido por StringIndexer para Ingrediente
model = ingred_indexer.fit(data3)
print(list(enumerate(model.labels)))

In [ ]:
# 3) SPLIT + ENTRENAMIENTO
# Este bloque hace la partición de datos y el entrenamiento de un árbol de decisión en Spark MLlib.
train, test = data_feat.randomSplit([0.8, 0.2], seed=123)
# randomSplit([0.8, 0.2]) divide aleatoriamente data_feat (que ya tiene features y label) en:
# 80% para entrenamiento (train)
# 20% para prueba (test)
# seed=123 fija la semilla para que la partición sea reproducible (si vuelves a ejecutar, cae igual).
from pyspark.ml.classification import DecisionTreeClassifier
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=6, maxBins=256, impurity="gini")
# Importas el clasificador de árbol de decisión.
# featuresCol="features" / labelCol="label": le dices al modelo dónde están las entradas y la etiqueta.
# maxDepth=6: profundidad máxima del árbol (controla complejidad).
# Más grande ⇒ modelo más expresivo, pero riesgo de sobreajuste.
# Más pequeño ⇒ modelo más simple, quizá menor accuracy pero mejor generalización.
# maxBins=256: nº de “cubetas” para decidir divisiones en features categóricas.
# Debe ser ≥ al nº de categorías de cada feature categórica.
# En tu caso, IngredienteIndex puede tener muchas categorías; por eso lo ponemos alto para evitar el error “requires maxBins ≥ numCategories”.
# impurity="gini": criterio para elegir los splits (puedes usar "entropy" también).
# Ambos funcionan para multiclase; gini suele ser un poco más rápido.
dt_model = dt.fit(train)
# Entrena el árbol solo con train:
# Busca, nodo a nodo, las divisiones que más reduzcan la impureza (gini) en la etiqueta.
# Se detiene cuando llega a maxDepth, no hay ganancia suficiente, o no hay datos para seguir.
# Devuelve un DecisionTreeClassificationModel (con los nodos, reglas y importancia de variables).

In [ ]:
# 4) PREDICCIÓN + EVALUACIÓN
pred = dt_model.transform(test)
# Aplica el modelo entrenado (dt_model) sobre el conjunto de prueba (test).
# Devuelve un DataFrame pred que contiene todas las columnas de test más las columnas generadas por el modelo:
# prediction: clase predicha (numérica).
# rawPrediction, probability (en árboles no siempre se usan, pero pueden aparecer).
pred.select("features","label","prediction").show(10, truncate=False)
# Muestra las primeras 10 filas con:
# features: vector de entrada.
# label: clase real (numérica).
# prediction: clase predicha (numérica).
# En tu actividad el mapeo es fijo:
# 0 = Seco, 1 = Graso, 2 = Mixto, 3 = Dañado/Teñido.
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
acc = evaluator.evaluate(pred)
print("Accuracy:", round(acc, 4))
# Accuracy = (# aciertos) / (total de ejemplos en test).
# Toma label (real) y prediction (modelo) y calcula qué proporción coincide.
# Resultado entre 0 y 1 (ej. 0.87 ⇒ 87% de acierto).

print("\nMatriz de confusión (label vs prediction):")
(pred.groupBy("label","prediction").count()
     .orderBy("label","prediction")
     .show(truncate=False))
# Agrupa por par (etiqueta real, predicción) y cuenta ocurrencias.
# La diagonal (label = prediction) son aciertos.
# Las celdas fuera de la diagonal son errores (qué clases se confunden).
# Si quieres verla como una tabla pivote por filas/columnas:

+--------------------------+-----+----------+
|features                  |label|prediction|
+--------------------------+-----+----------+
|(6,[0,3],[2.0,5.81])      |3.0  |3.0       |
|(6,[1,3],[2.0,5.72])      |0.0  |0.0       |
|(6,[3,5],[5.07,1.0])      |0.0  |0.0       |
|(6,[3,5],[5.18,1.0])      |0.0  |0.0       |
|[0.0,0.0,2.0,5.3,0.0,1.0] |0.0  |0.0       |
|[0.0,1.0,0.0,6.37,1.0,0.0]|0.0  |0.0       |
|[0.0,1.0,1.0,5.39,0.0,1.0]|0.0  |0.0       |
|[0.0,1.0,1.0,5.43,0.0,0.0]|0.0  |0.0       |
|[0.0,1.0,1.0,5.54,0.0,1.0]|0.0  |0.0       |
|[0.0,1.0,1.0,5.7,0.0,1.0] |0.0  |0.0       |
+--------------------------+-----+----------+
only showing top 10 rows

Accuracy: 0.8923

Matriz de confusión (label vs prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|0.0  |0.0       |32   |
|1.0  |1.0       |11   |
|1.0  |3.0       |1    |
|2.0  |1.0       |1    |
|2.0  |2.0       |4    |
|2.0  |3.0       |1    |
|3.0  |1.0       |2    |
|3.0  |2.0       |2

1) Filas de predicción
+--------------------------+-----+----------+
|features                  |label|prediction|
+--------------------------+-----+----------+
|(6,[0,3],[2.0,5.81])      |3.0  |3.0       |
|(6,[1,3],[2.0,5.72])      |0.0  |0.0       |
|(6,[3,5],[5.07,1.0])      |0.0  |0.0       |
...


features es el vector con este orden fijo:
["IngredienteIndex","LimpiezaIndex","NutricionIndex","pH","SulfatosBin","SiliconaBin"].

A veces sale disperso (sparse): |(6,[0,3],[2.0,5.81])|

“6” = tamaño del vector.

Índices [0,3] tienen valores [2.0, 5.81].

Traducido: IngredienteIndex=2.0, pH=5.81, y las demás posiciones valen 0.0.

Ojo: 0.0 es un valor válido (ej. Bajo=0, No=0), no significa “faltante”.

label es la clase real y prediction la predicha por el árbol. Recuerda el mapeo:

0 = Seco, 1 = Graso, 2 = Mixto, 3 = Dañado/Teñido.

Ejemplos:

Fila 1: (6,[0,3],[2.0,5.81]) | label=3.0 | pred=3.0 → Correcto (Dañado/Teñido).

Fila 2: (6,[1,3],[2.0,5.72]) | label=0.0 | pred=0.0 → Correcto (Seco).

Filas densas como [0.0,1.0,1.0,5.7,0.0,1.0] muestran todas las 6 posiciones explícitas.

2) Accuracy: 0.8923

Exactitud global en test ≈ 89.23%.

Con los números de la matriz: 58 aciertos / 65 ejemplos ≈ 0.8923.

3) Matriz de confusión
label  pred  count
0.0    0.0   32    ← aciertos Seco
1.0    1.0   11    ← aciertos Graso
1.0    3.0    1    ← Graso → Dañado/Teñido
2.0    1.0    1    ← Mixto → Graso
2.0    2.0    4    ← aciertos Mixto
2.0    3.0    1    ← Mixto → Dañado/Teñido
3.0    1.0    2    ← Dañado/Teñido → Graso
3.0    2.0    2    ← Dañado/Teñido → Mixto
3.0    3.0   11    ← aciertos Dañado/Teñido


Aciertos (diagonal): 32 (Seco) + 11 (Graso) + 4 (Mixto) + 11 (Dañado/Teñido) = 58.

Totales por clase real y precisión por clase:

Seco (0): 32/32 = 100%

Graso (1): (11+1)=12 → 11/12 = 91.7%

Mixto (2): (1+4+1)=6 → 4/6 = 66.7%

Dañado/Teñido (3): (2+2+11)=15 → 11/15 = 73.3%

Dónde se confunde más:

Mixto (2) se confunde con Graso (1) y Dañado/Teñido (3).

Dañado/Teñido (3) se confunde algo con Graso (1) y Mixto (2).

Esto sugiere que, con las features actuales, Mixto y Dañado/Teñido quedan más cercanos en el espacio de decisión.

In [ ]:
# 5) Análisis rápido: importancia y reglas del árbol
print("\nImportancia de variables:")
for name, score in zip(feature_cols, dt_model.featureImportances.toArray()):
    print(f"  {name:>15}: {round(score, 4)}")
# dt_model.featureImportances devuelve un vector (normalmente sparse) con una importancia por posición de feature.
# toArray() lo convierte a un arreglo; zip(...) lo alinea con tus nombres en feature_cols.
# Cada score está en [0, 1] y la suma ≈ 1 (puede variar por redondeo).
# ¿Qué es “importancia”? En árboles, es la reducción total de impureza (gini/entropía) que aportaron los splits de esa variable, ponderada por
# cuántas filas pasan por esos nodos.
# Más alto ⇒ la variable se usó en más splits y/o hizo splits “buenos”.
# 0.0 ⇒ no se usó en ningún split.

# En tu caso (orden):
# feature_cols = [
#  "IngredienteIndex", "LimpiezaIndex", "NutricionIndex",
#  "pH", "SulfatosBin", "SiliconaBin"
# ]
# Si ves, por ejemplo, pH: 0.32 y IngredienteIndex: 0.40, puedes decir que Ingrediente y pH guiaron la mayor parte de las decisiones del árbol.

print("\nÁrbol aprendido:")
print(dt_model.toDebugString)
# toDebugString imprime el árbol de forma textual: nodos con condiciones “If (feature i ≤ umbral)” y hojas con “Predict: clase”.
# Los índices de feature son 0-based. Con tu vector:
# feature 0 → IngredienteIndex
# feature 1 → LimpiezaIndex
# feature 2 → NutricionIndex
# feature 3 → pH
# feature 4 → SulfatosBin
# feature 5 → SiliconaBin


Importancia de variables:
  IngredienteIndex: 0.6774
    LimpiezaIndex: 0.0298
   NutricionIndex: 0.1748
               pH: 0.0164
      SulfatosBin: 0.0231
      SiliconaBin: 0.0785

Árbol aprendido:
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_240c9f66ad65, depth=6, numNodes=27, numClasses=4, numFeatures=6
  If (feature 0 in {0.0,1.0,6.0})
   Predict: 0.0
  Else (feature 0 not in {0.0,1.0,6.0})
   If (feature 2 <= 1.5)
    If (feature 0 in {3.0,7.0})
     Predict: 1.0
    Else (feature 0 not in {3.0,7.0})
     If (feature 0 in {2.0,4.0})
      If (feature 1 <= 1.5)
       Predict: 3.0
      Else (feature 1 > 1.5)
       If (feature 4 <= 0.5)
        Predict: 3.0
       Else (feature 4 > 0.5)
        Predict: 1.0
     Else (feature 0 not in {2.0,4.0})
      If (feature 1 <= 0.5)
       Predict: 2.0
      Else (feature 1 > 0.5)
       If (feature 1 <= 1.5)
        Predict: 3.0
       Else (feature 1 > 1.5)
        Predict: 2.0
   Else (feature 2 > 1.5)
    If (feature 5

A) Importancia de variables
IngredienteIndex: 0.6774
LimpiezaIndex  : 0.0298
NutricionIndex : 0.1748
pH             : 0.0164
SulfatosBin    : 0.0231
SiliconaBin    : 0.0785


Suman ≈ 1.0, como debe ser.

IngredienteIndex (0.6774) domina: la elección del ingrediente explica ~68% de la reducción de impureza en el árbol.

Luego NutricionIndex (0.1748) y SiliconaBin (0.0785) aportan bastante.

Limpieza, Sulfatos y pH influyen menos en este modelo concreto.

Ojo didáctico: en árboles, una categórica con muchas categorías (como Ingrediente) puede quedar “favorecida” en importancias.

B) Árbol aprendido (reglas)

Mapa de índices → nombres de features en tu features:

feature 0 → IngredienteIndex

feature 1 → LimpiezaIndex (Bajo=0, Medio=1, Alto=2)

feature 2 → NutricionIndex (Bajo=0, Medio=1, Alto=2)

feature 3 → pH

feature 4 → SulfatosBin (No=0, Sí=1)

feature 5 → SiliconaBin (No=0, Sí=1)

Y el mapeo de clases (label/prediction):

0.0 = Seco, 1.0 = Graso, 2.0 = Mixto, 3.0 = Dañado/Teñido

Cómo leer los nodos

If (feature 0 in {0.0,1.0,6.0}) Predict: 0.0
Si IngredienteIndex está en ese conjunto de categorías, predice Seco directamente.
(Es un split categórico: “in {…}” = algunas categorías van por esa rama.)

If (feature 2 <= 1.5)
Si NutricionIndex ≤ 1.5 ⇒ es Bajo o Medio (no Alto).
Dentro:

If (feature 0 in {3.0,7.0}) Predict: 1.0
Para ciertos ingredientes (dos categorías concretas), predice Graso.

If (feature 0 in {2.0,4.0}) …
Para otros ingredientes, mira Limpieza y Sulfatos:

LimpiezaIndex ≤ 1.5 → Dañado/Teñido

LimpiezaIndex > 1.5 y SulfatosBin ≤ 0.5 (No) → Dañado/Teñido

LimpiezaIndex > 1.5 y SulfatosBin > 0.5 (Sí) → Graso

Else (feature 2 > 1.5)
NutricionIndex = Alto.

If (SiliconaBin ≤ 0.5) (No): usa pH y Ingrediente para decidir entre Graso / Mixto / Dañado.
Ej.: pH ≤ 5.56 y cierto ingrediente ⇒ Graso; si no, Mixto.
Con pH > 5.56, para algunos ingredientes: pH ≤ 5.775 ⇒ Graso; si no ⇒ Dañado/Teñido.

Else (SiliconaBin > 0.5) (Sí): predice Seco (muy coherente con la regla de generación del dataset: Nutrición Alta + Silicona Sí → Seco).

Idea clave

El árbol refleja relaciones que definiste al crear el dataset sintético:

Ingredientes “nutritivos” y Silicona=Sí empujan a Seco.

Limpieza Alta + Sulfatos=Sí empuja a Graso.

Ciertos ingredientes (p. ej. tipo keratina/colágeno en tus reglas originales) empujan a Dañado/Teñido.

pH afina decisiones cuando Nutrición es Alta y Silicona es No.

In [ ]:
# (Opcional) Comparar con Random Forest
# Ese bloque agrega un modelo de Random Forest para comparar contra tu árbol de decisión.
from pyspark.ml.classification import RandomForestClassifier
# Importa la clase RandomForestClassifier de Spark MLlib (clasificación supervisada).
# Random Forest = conjunto (ensamble) de muchos árboles; suele generalizar mejor que un árbol único
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200, maxDepth=8, maxBins=256, seed=123)
# Crea el estimador (configura el modelo, todavía no entrena):
# featuresCol="features" → columna de entrada con el vector que armaste con VectorAssembler.
# labelCol="label" → columna objetivo (0=Seco, 1=Graso, 2=Mixto, 3=Dañado/Teñido).
# numTrees=200 → cantidad de árboles en el bosque. Más árboles ↓varianza (hasta cierto punto) y ↑costo de cómputo.
# maxDepth=8 → profundidad máxima de cada árbol (controla complejidad/overfitting).
# maxBins=256 → nº de “bins” para discretizar/cotejar features categóricas (debe ser ≥ a la cantidad de categorías de cualquier
# feature, p. ej. IngredienteIndex).
# seed=123 → semilla para que los resultados sean reproducibles.
rf_model = rf.fit(train)
# Entrena el bosque con el conjunto train.
# Internamente, cada árbol se entrena sobre submuestras de filas y subconjuntos de variables (bagging + feature subsampling).
pred_rf = rf_model.transform(test)
# Aplica el modelo entrenado al conjunto test.
# Devuelve un DataFrame con columnas originales + columnas de predicción (prediction, probability, etc.).
acc_rf = evaluator.evaluate(pred_rf)
print("\n[RandomForest] Accuracy:", round(acc_rf, 4))


[RandomForest] Accuracy: 0.9385


ccuracy (exactitud) = proporción de aciertos en el conjunto de prueba.

0.9385 ⇒ ~93.85% de las predicciones del Random Forest fueron correctas.

Si tu test tenía ~65 ejemplos (como antes), eso equivale aprox. a 61/65 aciertos.

¿Por qué suele ser mejor que el árbol?

Random Forest promedia muchos árboles entrenados con submuestras (bagging + submuestreo de variables), lo que reduce la varianza y mejora la generalización. Por eso normalmente supera al árbol único (antes tenías ~0.8923).